In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
import mrs
import numpy as np
import pickle
import pandas as pd

In [ ]:
# load example problem and Pareto front
DATA_DIR = "../simulations"

SETUP = {
    "n_obj": 5,
    "n_con": 1,
    "n_items": 100,
    "n_selected": 25,
    "rho": 0.2,
    "exactly": True,
    "aspiration": np.array([14582, 11126,  7081,  9801,  8849]),
    "obj_names": ["obj_1", "obj_2", "obj_3", "obj_4", "obj_5"]
}

data = np.load(f'{DATA_DIR}/problems/items_{SETUP["n_obj"]}obj_{SETUP["n_items"]}items_{SETUP["rho"]}rho_allneg.npz')
items = data['items']
capacity = data['capacity']

RESULTS_DIR = "exact" if SETUP["exactly"] else "upper_bound"
SUB_DIR = "prefer" if SETUP["aspiration"] is not None else "no_prefer"
with open(f"{DATA_DIR}/results/{RESULTS_DIR}/{SUB_DIR}/eda_mode2_{SETUP['n_obj']}obj_{SETUP['n_items']}items_{SETUP['rho']}rho_allneg_withtime.pkl", "rb") as f:
    results = pickle.load(f)
pf_raw = results["converged_pf_table"][-1]
ps_raw = results["converged_pf_population_table"][-1]

In [ ]:
# compute mrs
OUTPUT_DIR = f"results/{SETUP["n_obj"]}obj_{SETUP["n_items"]}items_{SETUP["rho"]}rho"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
OBJ_COLORS = {
    "obj_1": "#4C78A8",   # blue
    "obj_2": "#F58518",   # orange
    "obj_3": "#54A24B",   # green
    "obj_4": "#E45756",   # red
    "obj_5": "#B279A2",   # purple
    "obj_6": "#FF9DA6",   # pink
    "obj_7": "#9D755D",   # brown
    "obj_8": "#2CB1BC",   # teal
}
density_threshold = 0.8

pf, kept_index = mrs.density_filter(pf_raw, density_threshold)
df, _ = mrs.pf_to_csv(pf, OUTPUT_DIR, SETUP["obj_names"])
mrs_params = mrs.get_mrs_params()
info = mrs.get_pf_info(df)
query_idx, query_pt = mrs.get_query_point(info["data"], SETUP["aspiration"], info["iqr"])
betas_raw, betas = mrs.compute_mrs_at_query(
        info["data"], 
        mrs_params["k"], 
        info["d"], 
        info["iqr"], 
        query_idx
    )
# mrs.save_mrs_results(betas_raw, betas, OUTPUT_DIR, info["obj_names"])
mrs.plot_bar_chart(
        betas, 
        info["obj_names"], 
        query_pt, 
        info["d"],
        OUTPUT_DIR, 
        obj_colors=OBJ_COLORS,
    )
radii_pct, radii_raw = mrs.compute_validity_radii(
        info["data"], 
        mrs_params["k"], 
        info["d"], 
        info["iqr"],
        query_idx, 
        mrs_params["epsilon"],
    )
mrs.plot_validity_bar_chart(
    radii_pct, 
    radii_raw, 
    info["obj_names"], 
    info["d"], 
    mrs_params["epsilon"],
    OUTPUT_DIR, 
    obj_colors=OBJ_COLORS,
)

In [ ]:
print(SETUP["aspiration"])
print(query_pt)

In [ ]:
# compute objective weights
from scipy.stats import percentileofscore
obj_weight = np.array([
    percentileofscore(pf[:,i], SETUP["aspiration"][i])
    for i in range(pf.shape[1])
])
obj_weight = obj_weight / np.sum(obj_weight)
print(obj_weight)

In [ ]:
# weighted average mrs
betas_weighted = betas @ obj_weight.T
betas_weighted

In [ ]:
change_i = np.argmax(betas_weighted)
# change = np.nanmin(radii_raw[change_i])
change = np.nanmean(radii_raw[change_i])


In [ ]:
target = query_pt.copy()
for j in range(len(query_pt)):
    if j != change_i:
        target[j] = query_pt[j] + betas_raw[change_i, j] * change
target[change_i] = query_pt[change_i] + change

candidates = np.flatnonzero(info["data"][:, change_i] > query_pt[change_i])
dists = np.abs((info["data"][candidates] - target) / info["iqr"]).sum(axis=1)
query_idx = int(candidates[np.argmin(dists)])
query_pt = info["data"][query_idx]
print(query_pt)

In [ ]:
# obtain mrs at new query point
betas_raw, betas = mrs.compute_mrs_at_query(
        info["data"], 
        mrs_params["k"], 
        info["d"], 
        info["iqr"], 
        query_idx
    )
mrs.plot_bar_chart(
        betas, 
        info["obj_names"], 
        query_pt, 
        info["d"],
        OUTPUT_DIR, 
        obj_colors=OBJ_COLORS,
    )
radii_pct, radii_raw = mrs.compute_validity_radii(
        info["data"], 
        mrs_params["k"], 
        info["d"], 
        info["iqr"],
        query_idx, 
        mrs_params["epsilon"],
    )
mrs.plot_validity_bar_chart(
    radii_pct, 
    radii_raw, 
    info["obj_names"], 
    info["d"], 
    mrs_params["epsilon"],
    OUTPUT_DIR, 
    obj_colors=OBJ_COLORS,
)


In [ ]:
# plot query-point trajectory from mrs_loop.py history
import math
import itertools
import matplotlib.pyplot as plt

RUN_DIR = Path("results/5obj_100items_0.2rho/loop/test1")
SHOW_PF = True

history = pd.read_csv(RUN_DIR / "history.csv")
summary = pd.read_csv(RUN_DIR / "outer_summary.csv")
objective_names = [c.removeprefix("query_") for c in history.columns if c.startswith("query_obj")]
n_obj = len(objective_names)
query_cols = [f"query_{n}" for n in objective_names]
aspi_cols = [f"aspi_{n}" for n in objective_names]
final_cols = [f"final_{n}" for n in objective_names]

# trajectory per outer iteration: recorded steps + the point reached by the last move
trajectories = {}
for outer, grp in history.groupby("outer"):
    traj = grp.sort_values("step")[query_cols].to_numpy()
    final_pt = summary.loc[summary["outer"] == outer, final_cols].to_numpy()[0]
    if not np.array_equal(traj[-1], final_pt):
        traj = np.vstack([traj, final_pt])
    trajectories[outer] = traj

outer_colors = plt.cm.tab10(np.arange(len(trajectories)) % 10)

# zoom each panel to the visited points (+ aspirations) instead of the whole PF
visited = np.vstack(list(trajectories.values()) + [summary[aspi_cols].to_numpy()])
pad = 0.15 * (visited.max(axis=0) - visited.min(axis=0))
lims = np.stack([visited.min(axis=0) - pad, visited.max(axis=0) + pad], axis=1)

objective_pairs = list(itertools.combinations(range(n_obj), 2))
n_plots = len(objective_pairs)
cols = min(5, n_plots)
rows = math.ceil(n_plots / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
axes = np.atleast_1d(axes).ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    for color, (outer, traj) in zip(outer_colors, trajectories.items()):
        if SHOW_PF:
            pf_outer = pd.read_csv(RUN_DIR / f"outer_{outer:02d}" / "pf.csv").to_numpy()
            ax.plot(pf_outer[:, a], pf_outer[:, b], "o", color=color, alpha=0.04, markersize=2)
        aspi = summary.loc[summary["outer"] == outer, aspi_cols].to_numpy()[0]
        ax.plot(aspi[a], aspi[b], "*", color=color, markeredgecolor="black",
                markersize=12, label=f"aspiration (outer {outer})")
        ax.plot(traj[:, a], traj[:, b], "-o", color=color, alpha=0.9, markersize=5,
                label=f"query path (outer {outer})")
        for s, (x, y) in enumerate(traj[:, [a, b]]):
            ax.annotate(str(s), (x, y), textcoords="offset points", xytext=(3, 3), fontsize=7, color=color)
    ax.set_xlim(lims[a])
    ax.set_ylim(lims[b])
    ax.set_xlabel(objective_names[a])
    ax.set_ylabel(objective_names[b])

for ax in axes[n_plots:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="upper left",
    bbox_to_anchor=(1.0, 1.0),
)
fig.tight_layout()
fig.savefig(RUN_DIR / "query_trajectory.png", bbox_inches="tight")
plt.show()

In [ ]:
# objective weights at every step (stacked bars); weights are fixed within an outer iteration
RUN_DIR = Path("results/5obj_100items_0.2rho/loop/test1")

history = pd.read_csv(RUN_DIR / "history.csv").sort_values(["outer", "step"]).reset_index(drop=True)
objective_names = [c.removeprefix("weight_") for c in history.columns if c.startswith("weight_")]
weights = history[[f"weight_{n}" for n in objective_names]].to_numpy()
x = np.arange(len(history))

fig, ax = plt.subplots(figsize=(max(6, 0.6 * len(history) + 2), 4.5))
bottom = np.zeros(len(history))
for i, name in enumerate(objective_names):
    ax.bar(x, weights[:, i], bottom=bottom, color=OBJ_COLORS[name], alpha=0.85,
           edgecolor="white", linewidth=0.5, label=name)
    for xi, (w, b0) in enumerate(zip(weights[:, i], bottom)):
        if w > 0.05:
            ax.text(xi, b0 + w / 2, f"{w:.2f}", ha="center", va="center", fontsize=7, color="white")
    bottom += weights[:, i]

# objective chosen to improve at each step
for xi, obj in enumerate(history["change_obj"]):
    ax.text(xi, 1.02, obj, ha="center", va="bottom", fontsize=7, rotation=90, color=OBJ_COLORS[obj])

# outer-iteration boundaries
for xi in np.flatnonzero(np.diff(history["outer"].to_numpy())) + 0.5:
    ax.axvline(xi, color="black", lw=1, ls="--")

ax.set_xticks(x)
ax.set_xticklabels([f"o{o}\ns{s}" for o, s in zip(history["outer"], history["step"])], fontsize=8)
ax.set_xlabel("outer iteration / inner step")
ax.set_ylabel("objective weight")
ax.set_ylim(0, 1.15)
ax.set_title("Objective weights per step (label above bar = objective improved)", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
fig.savefig(RUN_DIR / "obj_weights.png", bbox_inches="tight")
plt.show()

In [ ]:
# query point at each step (+ final point reached by the last move of each outer iteration)
RUN_DIR = Path("results/5obj_100items_0.2rho/loop/test1")

history = pd.read_csv(RUN_DIR / "history.csv")
summary = pd.read_csv(RUN_DIR / "outer_summary.csv")
objective_names = [c.removeprefix("query_") for c in history.columns if c.startswith("query_obj")]

steps = history[["outer", "step", "query_idx"] + [f"query_{n}" for n in objective_names]]
steps = steps.rename(columns={f"query_{n}": n for n in objective_names})
finals = summary[["outer", "final_query_idx"] + [f"final_{n}" for n in objective_names]]
finals = finals.rename(columns={"final_query_idx": "query_idx", **{f"final_{n}": n for n in objective_names}})
finals.insert(1, "step", "final")

query_table = pd.concat([steps, finals]).sort_values("outer", kind="stable").reset_index(drop=True)
print(query_table.to_string(index=False))

Inner vs outer loop

In [3]:
# inner vs outer loop: query-point trajectories, size of query-point moves, validity radii
import math
import itertools
import matplotlib.pyplot as plt

CMP_DIR = Path("results/5obj_100items_0.2rho_allneg_1600seed/inner_vs_outer/chain")
SHOW_PF = True

with open(CMP_DIR / "comparison.pkl", "rb") as f:
    cases = pickle.load(f)
pf_orig = pd.read_csv(CMP_DIR / "original_front" / "pf.csv")
objective_names = list(pf_orig.columns)
n_obj = len(objective_names)
# moves on both fronts are measured in units of the original front's IQR
iqr_orig = mrs.get_pf_info(pf_orig)["iqr"]

# inner trajectory: start of every case + the point reached by the last inner move
inner_traj = np.array([c["inner"]["query_pt"] for c in cases])
if cases[-1]["inner"]["new_query_pt"] is not None:
    inner_traj = np.vstack([inner_traj, cases[-1]["inner"]["new_query_pt"]])

# ---------------------------------------------------------------- trajectories
case_colors = plt.cm.tab10(np.arange(len(cases)) % 10)
pts = [inner_traj] + [np.array([c["aspiration"], c["outer"]["query_pt"]]) for c in cases]
pts += [c["outer"]["new_query_pt"][None] for c in cases if c["outer"]["new_query_pt"] is not None]
visited = np.vstack(pts)
pad = 0.15 * (visited.max(axis=0) - visited.min(axis=0))
lims = np.stack([visited.min(axis=0) - pad, visited.max(axis=0) + pad], axis=1)

objective_pairs = list(itertools.combinations(range(n_obj), 2))
n_plots = len(objective_pairs)
cols = min(5, n_plots)
rows = math.ceil(n_plots / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
axes = np.atleast_1d(axes).ravel()
for ax, (a, b) in zip(axes, objective_pairs):
    if SHOW_PF:
        ax.plot(pf_orig.iloc[:, a], pf_orig.iloc[:, b], "o", color="grey", alpha=0.04, markersize=2)
    ax.plot(inner_traj[:, a], inner_traj[:, b], "-o", color="black", markersize=5, label="inner loop path")
    for s, (x, y) in enumerate(inner_traj[:, [a, b]]):
        ax.annotate(str(s), (x, y), textcoords="offset points", xytext=(3, 3), fontsize=7)
    for color, c in zip(case_colors, cases):
        start, new = c["outer"]["query_pt"], c["outer"]["new_query_pt"]
        # snap from the aspiration onto the case's new front
        ax.plot([c["aspiration"][a], start[a]], [c["aspiration"][b], start[b]], ":", color=color)
        ax.plot(start[a], start[b], "s", color=color, markersize=5)
        if new is not None:
            ax.annotate("", xy=(new[a], new[b]), xytext=(start[a], start[b]),
                        arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
            ax.plot(new[a], new[b], "^", color=color, markersize=6,
                    label=f"outer step (case {c['case']})")
    ax.set_xlim(lims[a])
    ax.set_ylim(lims[b])
    ax.set_xlabel(objective_names[a])
    ax.set_ylabel(objective_names[b])
for ax in axes[n_plots:]:
    ax.set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.suptitle("black: inner path (numbers = inner steps); square -> triangle: one outer step per case "
             "(dotted = snap from aspiration to the new front)", y=1.02)
fig.tight_layout()
fig.savefig(CMP_DIR / "query_trajectory_inner_vs_outer.png", bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------- move size and validity radii
rows_ = []
for c in cases:
    for mode in ("inner", "outer"):
        r = c[mode]
        finite = r["radii_raw"][np.isfinite(r["radii_raw"])]
        move = (np.full(n_obj, np.nan) if r["new_query_pt"] is None
                else r["new_query_pt"] - r["query_pt"])
        rows_.append({
            "case": c["case"], "mode": mode, "change_obj": r["change_obj"],
            "step_size": r["change"],
            "move_L1_iqr": np.abs(move / iqr_orig).sum(),
            "move_L2_iqr": np.linalg.norm(move / iqr_orig),
            "radii_median": np.median(finite),
            "radii_mean": finite.mean(),
            "radii_max": finite.max(),
            **{f"d_{n}": v for n, v in zip(objective_names, move)},
        })
moves = pd.DataFrame(rows_)
print(moves.round(2).to_string(index=False))

# radii compared entry by entry: same (moved objective, responding objective) pair in both loops
radii_cmp = []
for c in cases:
    ri, ro = c["inner"]["radii_raw"], c["outer"]["radii_raw"]
    both = np.isfinite(ri) & np.isfinite(ro)
    radii_cmp.append({"case": c["case"], "n_pairs": int(both.sum()),
                      "outer_larger": int((ro[both] > ri[both]).sum()),
                      "median_log_ratio_outer/inner": np.median(np.log(ro[both] / ri[both]))})
radii_cmp = pd.DataFrame(radii_cmp)
print()
print(radii_cmp.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ["move_L1_iqr", "step_size", "radii_median"],
                          ["query-point move (L1, original-front IQR units)",
                           "step size (smallest finite radius of chosen objective)",
                           "median finite validity radius"]):
    wide = moves.pivot(index="case", columns="mode", values=col)
    wide.plot.bar(ax=ax, color={"inner": "black", "outer": "tab:orange"}, rot=0)
    ax.set_title(title, fontsize=10)
fig.tight_layout()
fig.savefig(CMP_DIR / "move_and_radii_inner_vs_outer.png", bbox_inches="tight")
plt.show()

# summary; case 1 is excluded because both loops are identical there
later = moves[moves["case"] > 1]
for col, label in [("move_L1_iqr", "query-point move"), ("radii_median", "median validity radius")]:
    w = later.pivot(index="case", columns="mode", values=col)
    n_outer = int((w["outer"] > w["inner"]).sum())
    print(f"{label}: outer > inner in {n_outer}/{len(w)} cases; "
          f"median inner = {w['inner'].median():.2f}, median outer = {w['outer'].median():.2f}")


 case  mode change_obj  step_size  move_L1_iqr  move_L2_iqr  radii_median  radii_mean  radii_max  d_obj_1  d_obj_2  d_obj_3  d_obj_4  d_obj_5
    1 inner      obj_1    1084.44         2.48         1.33       1786.47     6553.62   74426.51   1458.0   -125.0   -574.0   -568.0   -666.0
    1 outer      obj_1    1084.44         2.48         1.33       1786.47     6553.62   74426.51   1458.0   -125.0   -574.0   -568.0   -666.0
    2 inner      obj_4     251.56         0.86         0.55        868.81     1615.80    7619.55    -94.0   -111.0   -134.0    694.0    139.0
    2 outer      obj_3     318.49         0.69         0.33       1058.22     1874.13    7485.89   -120.0   -180.0    323.0   -177.0    154.0
    3 inner      obj_3     564.32         0.39         0.20       1348.04    14992.48  138449.26   -113.0   -129.0    205.0    -45.0     46.0
    3 outer      obj_3     216.12         0.49         0.27        522.53     1299.44   13141.48    123.0     76.0    221.0     -5.0   -265.0
    4 

In [4]:
# objective weights at every query point of the inner and outer loops
# each point is weighted by its percentile on the front its loop uses:
# inner -> original front; outer -> the case's new front (case 1 reuses the original front)
import matplotlib.pyplot as plt
from mrs_loop import compute_obj_weight

OBJ_COLORS = {
    "obj_1": "#4C78A8",   # blue
    "obj_2": "#F58518",   # orange
    "obj_3": "#54A24B",   # green
    "obj_4": "#E45756",   # red
    "obj_5": "#B279A2",   # purple
    "obj_6": "#FF9DA6",   # pink
    "obj_7": "#9D755D",   # brown
    "obj_8": "#2CB1BC",   # teal
}
CMP_DIR = Path("results/5obj_100items_0.2rho_allneg_1600seed/inner_vs_outer/chain")

with open(CMP_DIR / "comparison.pkl", "rb") as f:
    cases = pickle.load(f)
pf_orig = pd.read_csv(CMP_DIR / "original_front" / "pf.csv")
objective_names = list(pf_orig.columns)

def front_for(case, mode):
    if mode == "inner" or case == 1:
        return pf_orig.to_numpy()
    return pd.read_csv(CMP_DIR / f"case_{case:02d}" / "outer" / "pf.csv").to_numpy()

weight_rows = []
for c in cases:
    for mode in ("inner", "outer"):
        r = c[mode]
        pf_mode = front_for(c["case"], mode)
        points = {"used": None, "query": r["query_pt"], "moved": r["new_query_pt"]}
        for point, pt in points.items():
            if point == "used":
                w = r["obj_weight"]
            elif pt is None:
                w = np.full(len(objective_names), np.nan)
            else:
                w = compute_obj_weight(pf_mode, pt)
            weight_rows.append({"case": c["case"], "mode": mode, "point": point,
                                "change_obj": r["change_obj"] if point == "query" else "",
                                **dict(zip(objective_names, w))})
query_weights = pd.DataFrame(weight_rows)
print("used = weight the step was driven by; query = weight at the start point; moved = weight at the new point")
print(query_weights.round(3).to_string(index=False))
query_weights.to_csv(CMP_DIR / "query_point_weights.csv", index=False)

# stacked bars: for each case, [used, query, moved] for each loop
fig, axes = plt.subplots(2, 1, figsize=(max(8, 1.5 * len(cases) + 2), 8), sharey=True)
for ax, mode in zip(axes, ("inner", "outer")):
    sub = query_weights[query_weights["mode"] == mode].reset_index(drop=True)
    x = sub["case"].to_numpy() + sub["point"].map({"used": -0.27, "query": 0.0, "moved": 0.27}).to_numpy()
    bottom = np.zeros(len(sub))
    for name in objective_names:
        vals = sub[name].fillna(0).to_numpy()
        ax.bar(x, vals, width=0.25, bottom=bottom, color=OBJ_COLORS[name], alpha=0.85,
               edgecolor="white", linewidth=0.5, label=name)
        bottom += vals
    for xi, point in zip(x, sub["point"]):
        ax.text(xi, 1.01, point[0], ha="center", va="bottom", fontsize=7)
    for xi, obj in zip(x[sub["point"] == "query"], sub.loc[sub["point"] == "query", "change_obj"]):
        ax.text(xi, 1.07, obj, ha="center", va="bottom", fontsize=7, color=OBJ_COLORS[obj])
    ax.set_xticks(sorted(sub["case"].unique()))
    ax.set_xlabel("case")
    ax.set_ylabel("objective weight")
    ax.set_ylim(0, 1.15)
    ax.set_title(f"{mode} loop: u = weight used, q = at query point, m = at moved point "
                 f"(coloured label = objective improved)", fontsize=10)
axes[0].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
fig.tight_layout()
fig.savefig(CMP_DIR / "query_point_weights.png", bbox_inches="tight")
plt.show()


used = weight the step was driven by; query = weight at the start point; moved = weight at the new point
 case  mode point change_obj  obj_1  obj_2  obj_3  obj_4  obj_5
    1 inner  used             0.275  0.144  0.116  0.188  0.276
    1 inner query      obj_1  0.241  0.185  0.111  0.173  0.290
    1 inner moved             0.460  0.184  0.057  0.094  0.205
    1 outer  used             0.275  0.144  0.116  0.188  0.276
    1 outer query      obj_1  0.241  0.185  0.111  0.173  0.290
    1 outer moved             0.460  0.184  0.057  0.094  0.205
    2 inner  used             0.275  0.144  0.116  0.188  0.276
    2 inner query      obj_4  0.460  0.184  0.057  0.094  0.205
    2 inner moved             0.410  0.149  0.041  0.194  0.206
    2 outer  used             0.369  0.143  0.075  0.139  0.274
    2 outer query      obj_3  0.348  0.144  0.065  0.162  0.280
    2 outer moved             0.336  0.121  0.102  0.135  0.306
    3 inner  used             0.275  0.144  0.116  0.188  0.276

In [9]:
# inner-loop cycle: the steps between the revisited points (B -> C -> ... -> B)
CMP_DIR = Path("results/5obj_100items_0.2rho_allneg_1400seed/inner_vs_outer/seed1400")
N_CANDIDATES = 3   # closest candidates to each target, to see why the step lands where it does

with open(CMP_DIR / "comparison.pkl", "rb") as f:
    cases = pickle.load(f)
pf_orig = pd.read_csv(CMP_DIR / "original_front" / "pf.csv")
objective_names = list(pf_orig.columns)
info_orig = mrs.get_pf_info(pf_orig)

inner_steps = [c["inner"] for c in cases]
query_idxs = [r["query_idx"] for r in inner_steps]
# first step whose new point was already visited (the run may keep cycling after it)
end = next((k for k, r in enumerate(inner_steps) if r["new_query_idx"] in query_idxs[:k + 1]), None)
if end is None:
    print("no cycle in the inner loop of this run")
else:
    last_new = inner_steps[end]["new_query_idx"]
    start = query_idxs.index(last_new)
    cycle = inner_steps[start:end + 1]
    labels = {r["query_idx"]: chr(ord("B") + i) for i, r in enumerate(cycle)}
    path = " -> ".join(labels[i] for i in query_idxs[start:end + 1] + [last_new])
    print(f"inner loop cycle (cases {cases[start]['case']}-{cases[end]['case']}): {path}")
    print("points:", ", ".join(f"{l} = idx {i}" for i, l in labels.items()))
    print(f"weights (fixed in the inner loop): {np.round(cycle[0]['obj_weight'], 3)}")

    for case, r in zip(cases[start:end + 1], cycle):
        frm, to = labels[r["query_idx"]], labels.get(r["new_query_idx"], "?")
        print(f"\n=== case {case['case']}: {frm} -> {to}, improve {r['change_obj']} by {r['change']:.2f} ===")
        step = pd.DataFrame(
            [r["query_pt"], r["betas_weighted"], r["target"], r["new_query_pt"],
             r["new_query_pt"] - r["query_pt"]],
            index=[f"query ({frm})", "weighted betas", "target", f"new query ({to})", "new - query"],
            columns=objective_names,
        )
        print(step.round(3).to_string())

        # candidates = original-front points that improve the chosen objective, ranked by
        # IQR-scaled L1 distance to the target (the rule move_query uses)
        data = info_orig["data"]
        cand = np.flatnonzero(data[:, r["change_i"]] > r["query_pt"][r["change_i"]])
        dists = np.abs((data[cand] - r["target"]) / info_orig["iqr"]).sum(axis=1)
        order = np.argsort(dists)[:N_CANDIDATES]
        nearest = pd.DataFrame(data[cand[order]], columns=objective_names)
        nearest.insert(0, "dist_to_target", dists[order])
        nearest.insert(0, "point", [labels.get(i, "") for i in cand[order]])
        nearest.insert(0, "idx", cand[order])
        print(f"closest {N_CANDIDATES} of {cand.size} candidates to the target:")
        print(nearest.round(3).to_string(index=False))


inner loop cycle (cases 2-3): B -> C -> B
points: B = idx 1495, C = idx 1881
weights (fixed in the inner loop): [0.129 0.251 0.256 0.241 0.124]

=== case 2: B -> C, improve obj_1 by 100.38 ===
                   obj_1      obj_2      obj_3      obj_4     obj_5
query (B)       7870.000  10883.000  14102.000  13201.000  7950.000
weighted betas    -0.259     -0.362     -0.261     -0.431    -0.352
target          7970.375  10841.885  14076.277  13163.769  7931.124
new query (C)   8045.000  10832.000  13971.000  13289.000  7983.000
new - query      175.000    -51.000   -131.000     88.000    33.000
closest 3 of 4686 candidates to the target:
 idx point  dist_to_target  obj_1   obj_2   obj_3   obj_4  obj_5
1881     C           0.283 8045.0 10832.0 13971.0 13289.0 7983.0
1960                 0.484 8081.0 10597.0 13894.0 13083.0 7968.0
1747                 0.491 7988.0 10986.0 13912.0 12916.0 8015.0

=== case 3: C -> B, improve obj_3 by 115.60 ===
                   obj_1      obj_2      obj_3

In [5]:
# aspiration, query point, target and new query point at every step of each loop
CMP_CSV = Path("results/5obj_100items_0.2rho_allneg_1600seed/inner_vs_outer/chain/comparison.csv")

comparison = pd.read_csv(CMP_CSV)
objective_names = [c.removeprefix("query_") for c in comparison.columns if c.startswith("query_obj")]
point_prefixes = {"aspiration": "aspi_", "query": "query_", "target": "target_",
                  "new query": "new_query_pt_"}

for mode in ("inner", "outer"):
    print(f"\n################ {mode} loop ################")
    for _, row in comparison[comparison["mode"] == mode].sort_values("case").iterrows():
        print(f"\n--- step {row['case']}: improve {row['change_obj']} by {row['change']:.2f} "
              f"(query idx {row['query_idx']} -> {row['new_query_idx']}) ---")
        points = pd.DataFrame(
            {label: [row[f"{prefix}{n}"] for n in objective_names]
             for label, prefix in point_prefixes.items()},
            index=objective_names,
        ).T
        print(points.round(1).to_string())



################ inner loop ################

--- step 1: improve obj_1 by 1084.44 (query idx 4087 -> 7017) ---
              obj_1   obj_2   obj_3    obj_4    obj_5
aspiration  12329.0  6994.0  9638.0  10307.0  13385.0
query       12240.0  7327.0  9661.0  10303.0  13603.0
target      13324.4  6762.6  8942.9   9595.0  12957.5
new query   13698.0  7202.0  9087.0   9735.0  12937.0

--- step 2: improve obj_4 by 251.56 (query idx 7017 -> 6934) ---
              obj_1   obj_2   obj_3    obj_4    obj_5
aspiration  12329.0  6994.0  9638.0  10307.0  13385.0
query       13698.0  7202.0  9087.0   9735.0  12937.0
target      13613.6  7029.5  8949.0   9986.6  12830.7
new query   13604.0  7091.0  8953.0  10429.0  13076.0

--- step 3: improve obj_3 by 564.32 (query idx 6934 -> 6804) ---
              obj_1   obj_2   obj_3    obj_4    obj_5
aspiration  12329.0  6994.0  9638.0  10307.0  13385.0
query       13604.0  7091.0  8953.0  10429.0  13076.0
target      13513.4  6952.1  9517.3  10319.7  12825.7